# 05 - Collections, Regex, Datetime and Data Formats

**Level:** Intermediate  |  **Track:** Python  |  **Python:** 3.14

this notebook Python of **"batteries included"** taqat of dil is. Jo cheezein
tum haath from `dict`/`list` from banate is, stinsided library me unke **fast, tested,
C-level** versions maujood are. Interview me yahi cheezein ginti are, and
production code me yahi din bachati are.

---

## This notebook me what seekhenge

**A. `collections` module (specialised containers)**
- `Counter` - counting of aftershah: `most_common`, `elements`, `+ - & |`, `update`, `subtract`, `total`
- `defaultdict` - missing key but khud value bana gives is (`list` / `int` / `set`) + factory gotcha
- `deque` - dono taraf from O(1) push/pop, `rotate`, `maxlen` sliding window, `list` from performance comparison
- `namedtuple` - halka immutable record: `_fields`, `_replace`, `_asdict`, `defaults`
- `OrdelayedDict` - Python 3.7+ me `dict` ordelayed is, phir this **kab** kaam aata is? (`move_to_end`, ordelay-sensitive equality)
- `ChainMap` - config layering (CLI > env > file > defaults) without merge of

**B. `heapq` (priority queue / heap)**
- `heappush`, `heappop`, `heapify`, `nlargest`, `nsmallest`, `heapreplace`, `heappushpop`
- min-heap then max-heap banane of **negate trick**
- tuple-based priority queue + tie-breaking counter (unordelayable items of hal)

**C. `bisect` (sorted list but withoutry search)**
- `insort`, `bisect_left` vs `bisect_right`, grade/tax-slab lookup, sorted list maintain to do

**D. `re` - Regular Expressions (deep dive)**
- literals, character classes, predefined classes (`\d \w \s` + negations)
- quantifiers: greedy vs lazy vs possessive
- anchors: `^ $ \b \B`
- groups, non-capturing `(?:...)`, named `(?P<name>...)`, alternation, backreferences
- lookahead / lookbehind (positive + negative)
- flags: `IGNORECASE`, `MULTILINE`, `DOTALL`, `VERBOSE`
- `match` vs `search` vs `fullmatch` vs `findall` vs `finditer` vs `sub` vs `subn` vs `split`
- compiled patterns + speed, **catastrophic backtracking** warning
- real validations: email, Pakistani mobile number, password strength, log parsing, date extraction, HTML-tag caution

**E. `datetime` / `zoneinfo` / `time`**
- `datetime`, `date`, `time`, `timedelta` arithmetic
- `strftime` / `strptime` directive table, ISO parsing
- naive vs timezone-aware, `timezone.utc`, `ZoneInfo("Asia/Karachi")`
- `time.perf_counter`, `time.monotonic`, `time.sleep`
- age / duration calculators

**F. Data formats**
- `json`: `dumps`/`loads`, `indent`, `sort_keys`, `default=`, custom `JSONEncodelay`, non-str keys gotcha
- `csv` (via `io.StringIO`): `readelay`, `DictReadelay`, `writer`, `DictWriter`, delimiters, quoting, `newline=''`
- `pickle` vs `json` (**security warning**)
- `base64`: bytes <-> str
- `dataclass` -> `dict` (`dataclasses.asdict`) -> JSON round-trip

**G.** Common Mistakes, 5 Real-World problems, 6 LeetCode problems, Interview Q&A, Practice Exercises, Cheat-sheet.

---

## Prerequisites
- Python basics: `list`, `dict`, `set`, `tuple`, loops, functions
- Comprehensions and `lambda`
- Classes of basic samajh (`dataclass` section of for)
- Big-O of exactly basic samajh (O(1) vs O(n) vs O(log n))

> **Tip:** har code cell then khud chalao, values badal kar see. Regex sirf parh kar samajh not aata - todna parta is.

In [ ]:
# ------------------------------------------------------------------
# Setup cell - poore notebook of imports yahin, ek dafa.
# (Notebook top-then-bottom runs is, is for first imports.)
# ------------------------------------------------------------------
import sys
import re                 # regular expressions
import json               # JSON serialization
import csv                # CSV read/write
import io                 # StringIO - memory me "file"
import base64             # bytes <-> ascii text
import pickle             # Python object serialization (KHATARNAAK - neeche parhna)
import heapq              # heap / priority queue
import bisect             # sorted list but withoutry search
import random
import tempfile
import os
import textwrap
import time as time_mod   # 'time' naam datetime.time from clash does is -> alias

from collections import Counter, defaultdict, deque, namedtuple, OrdelayedDict, ChainMap
from datetime import datetime, date, time as dtime, timedelta, timezone
from dataclasses import dataclass, field, asdict, fields as dc_fields
from time import perf_counter, monotonic, sleep

random.seed(42)           # deterministic output taake results repeatable hon

print("Python version :", sys.version.split()[0])
print("Imports ready   : collections, heapq, bisect, re, datetime, json, csv, pickle, base64")

---

# A1. `Counter` - counting of aftershah

## what is?
`Counter` ek **`dict` of subclass** is jo "cheez -> kitni dafa aayi" store does is.
Yani `dict[hashable, int]`, magar bohat si counting-specific superpowers of sath.

## why / kahan?
- Word frequency, log me status-code counting, votes ginna
- "Top K most frequent" those sawaal (LeetCode 347, 692 - neeche karenge)
- Anagram check, multiset (bag) arithmetic, inventory add/subtract

## Syntax
```python
from collections import Counter

Counter()                        # khaali
Counter("mississippi")           # iterable from
Counter(["a", "b", "a"])         # list from
Counter({"a": 3, "b": 1})        # mapping from
Counter(a=3, b=1)                # keyword from
```

## Sabse ahem baat (interview me poocha jata is)
`Counter` me **missing key error not gives** - `0` return does is:
```python
c = Counter("abc")
c["z"]      # 0  (Keyerror not)
```
but dhyan rakho: this **key add not does** (`defaultdict` of bar-aks). Read to do from
`c` me `"z"` shaamil not hoga.

## `most_common(n)`
- `n` do then top-n `(element, count)` tuples, **descending count** me
- `n` no do then saare, sorted
- **Ties of ordelay = insertion ordelay** (Python 3.7+), guaranteed lexicographic not.
  Is for LeetCode 692 (Top K Frequent **Words**) me alag from sorting should is is.

In [ ]:
# ------------------------------------------------------------------
# Counter: banane of tareeqe and most_common
# ------------------------------------------------------------------
text = "the quick brown fox jumps over the lazy dog the fox barks"
words = text.split()

wc = Counter(words)                 # list of words -> frequency map
print("Counter object :", wc)
print("type           :", type(wc).__mro__[:2])   # dict of subclass is
print()

# --- Access -------------------------------------------------------
print("wc['the']      :", wc["the"])        # 3
print("wc['zebra']    :", wc["zebra"])      # 0  <-- Keyerror not!
print("'zebra' in wc  :", "zebra" in wc)    # False - read to do from key BANI not
print()

# --- most_common --------------------------------------------------
print("top 3          :", wc.most_common(3))
print("sabse kam that :", wc.most_common()[-1])   # last element = least common
print()

# --- total() (Python 3.10+) : saare counts of jama --------------
print("total()        :", wc.total(), "| len(words) =", len(words))
assert wc.total() == len(words)      # har word ek dafa gina gaya

# --- character level counting -------------------------------------
cc = Counter("mississippi")
print("char counts    :", cc)
print("i kitni dafa   :", cc["i"])
assert cc["s"] == 4 and cc["i"] == 4 and cc["p"] == 2 and cc["m"] == 1

# --- Anagram check ek line me -------------------------------------
def is_anagram(a: str, b: str) -> bool:
    # Dono strings of letter-counts barabar hon then anagram are. O(n) time.
    return Counter(a.replace(" ", "").lower()) == Counter(b.replace(" ", "").lower())

print()
print("anagram 'listen'/'silent' :", is_anagram("listen", "silent"))
print("anagram 'hello'/'world'   :", is_anagram("hello", "world"))
assert is_anagram("Dirty Room", "Dormitory") is True
assert is_anagram("abc", "abd") is False

In [ ]:
# ------------------------------------------------------------------
# Counter arithmetic: + - & |   and   update / subtract / elements
# Counter then "multiset / bag" understand.
# ------------------------------------------------------------------
a = Counter(a=4, b=2, c=0, d=-2)
b = Counter(a=1, b=2, c=3, d=4)

print("a =", a)
print("b =", b)
print()

# 1) ADDITION: counts jama; result me sirf POSITIVE counts rehte are
print("a + b  (jama, non-positive drop)   :", a + b)
#   a:5, b:4, c:3, d:2   -> c=0+3=3 rakha gaya, some bhi <=0 not bacha

# 2) SUBTRACTION: ghatao; non-positive REMOVE is go are
print("a - b  (ghatao, non-positive drop) :", a - b)
#   a:3 sirf. b:0 -> hata, c:-3 -> hata, d:-6 -> hata

# 3) INTERSECTION (&) : har key of MINIMUM
print("a & b  (min per key)               :", a & b)

# 4) UNION (|) : har key of MAXIMUM
print("a | b  (max per key)               :", a | b)
print()

# --- Unary operators (Python 3.3+) --------------------------------
print("+a  (sirf positive counts rakho)   :", +a)   # a=4, b=2
print("-a  (negative then positive bana do) :", -a)   # d=2
print()

# --- update() vs subtract() : IN-PLACE, and inhe non-positive from farq not ---
stock = Counter(pen=10, book=5)
stock.update(Counter(pen=3, eraser=7))     # jama kar gives is (dict.update of tarah REPLACE not!)
print("update of after  :", stock)          # pen=13, book=5, eraser=7

stock.subtract(Counter(pen=20, book=1))    # ghatata is, NEGATIVE bhi rehne gives is
print("subtract of after:", stock)          # pen=-7  <-- this zinda is
print("negative stock  :", {k: v for k, v in stock.items() if v < 0})
print()

# GOTCHA: '-' operator negative drop does is, .subtract() not.
print("operator '-'  :", Counter(pen=10) - Counter(pen=20))     # Counter()  (khaali!)
tmp = Counter(pen=10)
tmp.subtract(Counter(pen=20))
print(".subtract()   :", tmp)                                   # Counter({'pen': -7... }) -> -10
assert (Counter(pen=10) - Counter(pen=20)) == Counter()
assert tmp["pen"] == -10
print()

# --- elements() : counts then wapas iterable me kholta is --------
inv = Counter(apple=3, banana=2, cherry=0, rotten=-1)
print("elements()    :", sorted(inv.elements()))
# NOTE: count <= 0 those elements() me exactly not aate (cherry, rotten gayab)
assert sorted(inv.elements()) == ["apple", "apple", "apple", "banana", "banana"]

# --- Chhota use-case: weighted random-free "expand" -------------
print("expanded str  :", "".join(sorted(Counter("aaabbc").elements())))

---

# A2. `defaultdict` - missing key but default khud ban jaye

## what is?
`dict` of subclass jise ek **`default_factory`** (zero-argument callable) diya jata is.
Jab tum aisi key maango jo maujood not, then `defaultdict` `default_factory()` chala kar
value bana gives is, **key insert kar gives is**, and wapas give gives is.

## why?
Grouping / accumulating code from `if key not in d:` boilerplate hata gives is.

```python
# old tareeqa
groups = {}
for k, v in pairs:
    if k not in groups:
        groups[k] = []
    groups[k].append(v)

# defaultdict
groups = defaultdict(list)
for k, v in pairs:
    groups[k].append(v)      # bas!
```

## Common factories
| Factory | `d[missing]` gives is | Kab |
|---|---|---|
| `list` | `[]` | grouping / one-then-many |
| `int` | `0` | counting / summing |
| `set` | `set()` | unique members |
| `dict` | `{}` | nested maps |
| `lambda: "N/A"` | `"N/A"` | custom default |
| `lambda: defaultdict(int)` | nested counter | 2-level counting |

## Interview tip
`defaultdict(int)` vs `Counter`: dono counting do are, magar `Counter` of sath
`most_common`, arithmetic, `elements` free milta is. `defaultdict(int)` general
accumulation (sum of prices, totals) of for behtar is.

In [ ]:
# ------------------------------------------------------------------
# defaultdict of teen sabse common patterns
# ------------------------------------------------------------------
students = [
    ("Lahore", "Ali"), ("Karachi", "Sana"), ("Lahore", "Bilal"),
    ("Islamabad", "Hina"), ("Karachi", "Usman"), ("Lahore", "Ali"),
]

# 1) defaultdict(list) -> GROUPING (one key -> many values)
by_city = defaultdict(list)
for city, name in students:
    by_city[city].append(name)          # no 'if city not in' needed
print("group by city :", dict(by_city))

# 2) defaultdict(int) -> COUNTING / SUMMING
sales = [("pen", 50), ("book", 300), ("pen", 20), ("bag", 900), ("book", 150)]
revenue = defaultdict(int)
for item, amount in sales:
    revenue[item] += amount             # 0 + amount pehli dafa
print("revenue       :", dict(revenue))

# 3) defaultdict(set) -> UNIQUE members (duplicates khud khatam)
unique_by_city = defaultdict(set)
for city, name in students:
    unique_by_city[city].add(name)
print("unique names  :", {k: sorted(v) for k, v in unique_by_city.items()})
print()

# 4) NESTED defaultdict -> 2-level counting (lambda zaroori: factory callable is)
#    Note: defaultdict(defaultdict(int)) GHALAT is - woh instance is, callable not.
matrix = defaultdict(lambda: defaultdict(int))
for city, name in students:
    matrix[city][name] += 1
print("nested counts :", {c: dict(v) for c, v in matrix.items()})
print()

# 5) defaultdict from GRAPH (adjacency list) - DSA me roz kaam aata is
edges = [(1, 2), (1, 3), (2, 4), (3, 4), (4, 5)]
graph = defaultdict(list)
for u, v in edges:
    graph[u].append(v)
    graph[v].append(u)                  # undirected
print("adjacency     :", dict(graph))

assert by_city["Lahore"] == ["Ali", "Bilal", "Ali"]
assert revenue["pen"] == 70
assert unique_by_city["Lahore"] == {"Ali", "Bilal"}
assert graph[4] == [2, 3, 5]

In [ ]:
# ------------------------------------------------------------------
# defaultdict of GOTCHAS - yahan log phaste are
# ------------------------------------------------------------------

# GOTCHA 1: Sirf PARHNE from bhi key BAN jati is (silent mutation!)
d = defaultdict(list)
print("shuru me        :", dict(d), "| len =", len(d))
_ = d["ghost"]                       # bas read done... but key insert is gayi
print("read of after    :", dict(d), "| len =", len(d))
assert "ghost" in d and d["ghost"] == []

# Bachne of tareeqa: .get() ya 'in' use do - this factory trigger not do
d2 = defaultdict(list)
print("d2.get('x')     :", d2.get("x"), "| len =", len(d2))     # None, len 0
print("'x' in d2       :", "x" in d2, "| len =", len(d2))       # False, len 0
assert len(d2) == 0
print()

# GOTCHA 2: default_factory None is then normal dict like Keyerror
d3 = defaultdict()                   # koi factory not di
print("default_factory :", d3.default_factory)
try:
    d3["nope"]
except Keyerror as e:
    print("Keyerror got   :", e, " <-- factory None hone but yehi is is")
print()

# GOTCHA 3: factory zero-argument hona should - key pass not is
#           "missing key of hisaab from default" should then __missing__ override do
class KeyAwareDict(dict):
    def __missing__(self, key):
        # Here key available is - defaultdict me not is
        value = f"<default for {key}>"
        self[key] = value
        return value

kad = KeyAwareDict()
print("KeyAwareDict    :", kad["speed"])
print()

# GOTCHA 4: defaultdict then function me pass to do -> caller of dict bhi mutate hoga
def count_chars(s, sink=None):
    sink = defaultdict(int) if sink is None else sink   # mutable default from bacho
    for ch in s:
        sink[ch] += 1
    return sink
print("count_chars     :", dict(count_chars("banana")))
print()

# GOTCHA 5: JSON / printing of time plain dict me convert do
nested = defaultdict(lambda: defaultdict(int))
nested["a"]["x"] = 1
try:
    print(json.dumps(nested))        # this chal jata is (dict subclass is)...
except Typeerror as e:
    print("json error      :", e)
# ...magar nested lambda factory PICKLE not is:
try:
    pickle.dumps(nested)
except Exception as e:
    print("pickle error    :", type(e).__name__, "->", e)
    print("Fix             : json/pickle from first dict() me convert do")

---

# A3. `deque` - double-ended queue (dono siron but O(1))

## what is?
"Deck" read (double-ended queue). this ek **doubly-linked list of blocks** is.
`list` of bar-aks, `deque` of **dono** sire O(1) are.

| Operation | `list` | `deque` |
|---|---|---|
| append right | O(1) amortized | **O(1)** |
| pop right | O(1) | **O(1)** |
| insert at 0 / pop at 0 | **O(n)** (sab shift) | **O(1)** |
| random index `x[i]` | **O(1)** | O(n) (middle me) |
| slicing `x[1:5]` | supported | **NOT supported** |

## why / kahan?
- **Queue / BFS** - `popleft()` exactly O(1) (`list.pop(0)` O(n) is)
- **Sliding window** - `maxlen` do, purane khud gir go are
- **Undo history / recent-N log** - fixed size buffer
- **Round-robin scheduling** - `rotate()`
- **Palindrome check** - dono siron from compare

## Syntax
```python
deque(iterable=(), maxlen=None)

d.append(x)      d.appendleft(x)
d.pop()          d.popleft()
d.extend(it)     d.extendleft(it)   # extendleft ULTA daalta is!
d.rotate(n)      # n>0 -> right shift, n<0 -> left shift
d.maxlen         # read-only
```

## Trade-off (interview)
`deque` random access of for bad is (middle element O(n)). if tum index from
bohat access do is then `list` behtar. if dono siron but kaam is then `deque`.

In [ ]:
# ------------------------------------------------------------------
# deque basics
# ------------------------------------------------------------------
dq = deque([2, 3, 4])
print("start          :", dq)

dq.append(5)          # right
dq.appendleft(1)      # left  <-- list me this insert(0, x) = O(n) is
print("after appends  :", dq)

print("pop()          :", dq.pop(), "->", dq)        # right from nikala
print("popleft()      :", dq.popleft(), "->", dq)    # left from nikala
print()

# extend vs extendleft (extendleft REVERSE ordelay me daalta is - classic gotcha)
d1 = deque([1, 2, 3])
d1.extend([4, 5])
print("extend([4,5])      :", d1)
d2 = deque([1, 2, 3])
d2.extendleft([4, 5])
print("extendleft([4,5])  :", d2, " <-- 5 first aaya (ulta!)")
assert list(d2) == [5, 4, 1, 2, 3]
print()

# rotate: circular shift
r = deque([1, 2, 3, 4, 5])
r.rotate(2)      # right shift 2  -> aakhri 2 ahead aa gaye
print("rotate(2)      :", r)
r.rotate(-2)     # wapas
print("rotate(-2)     :", r)
assert list(r) == [1, 2, 3, 4, 5]

# rotate of asli use: round-robin turn
players = deque(["Ali", "Sana", "Bilal"])
for turn in range(4):
    print(f"  turn {turn}: {players[0]} khelega")
    players.rotate(-1)          # left rotate = next player front but
print()

# deque me slicing not is - this deliberate error demo is
try:
    _ = dq[0:2]
except Typeerror as e:
    print("slicing error  :", e)
    print("Fix            : list(dq)[0:2]  ya  itertools.islice(dq, 0, 2)")
print("workaround     :", list(deque([1, 2, 3, 4]))[0:2])
print()

# Palindrome check dono siron from - deque of natural fit
def is_palindrome(s: str) -> bool:
    d = deque(ch for ch in s.lower() if ch.isalnum())
    while len(d) > 1:
        if d.popleft() != d.pop():       # dono taraf O(1)
            return False
    return True

print("palindrome 'A man, a plan, a canal: Panama' :", is_palindrome("A man, a plan, a canal: Panama"))
print("palindrome 'hello'                          :", is_palindrome("hello"))
assert is_palindrome("Madam, I'm Adam") is True
assert is_palindrome("python") is False

In [ ]:
# ------------------------------------------------------------------
# maxlen: khud-ba-khud bounded buffer (sliding window / recent-N)
# ------------------------------------------------------------------
last3 = deque(maxlen=3)
for x in [10, 20, 30, 40, 50]:
    last3.append(x)
    print(f"append {x:>2} -> {list(last3)}")     # 3 from zyada are hi left from gir jata is
assert list(last3) == [30, 40, 50]
print("maxlen         :", last3.maxlen, "(read-only property)")
print()

# maxlen + sliding window average (moving average) - streaming data of pattern
readings = [12, 15, 11, 20, 25, 22, 30, 28]
window = deque(maxlen=3)
mov_avg = []
for r in readings:
    window.append(r)
    mov_avg.append(round(sum(window) / len(window), 2))
print("readings       :", readings)
print("3-point moving :", mov_avg)
print()

# maxlen + appendleft -> doosri taraf from girta is
tail = deque(maxlen=3)
for x in "abcde":
    tail.appendleft(x)
print("appendleft w/ maxlen=3 :", list(tail), " <-- right from gira")
assert list(tail) == ["e", "d", "c"]
print()

# ------------------------------------------------------------------
# PERFORMANCE: list.pop(0) vs deque.popleft()  (queue simulation)
# ------------------------------------------------------------------
N = 60_000

lst = list(range(N))
t0 = perf_counter()
while lst:
    lst.pop(0)                  # O(n) har dafa -> total O(n^2)
t_list = perf_counter() - t0

dqq = deque(range(N))
t0 = perf_counter()
while dqq:
    dqq.popleft()               # O(1) har dafa -> total O(n)
t_deque = perf_counter() - t0

print(f"N = {N:,}")
print(f"list.pop(0)      : {t_list*1000:8.2f} ms   (O(n) per op)")
print(f"deque.popleft()  : {t_deque*1000:8.2f} ms   (O(1) per op)")
if t_deque > 0:
    print(f"deque ~{t_list/t_deque:.1f}x fast (machine but number badalta rahega)")
print()
print("Sabaq: BFS / queue / producer-consumer me list.pop(0) no use do -> deque use do.")

---

# A4. `namedtuple` - halka, immutable, self-documenting record

## what is?
`namedtuple` ek **class factory** is. this `tuple` of subclass banata is jisme
fields of **naam** are are. Memory `tuple` jitni (no `__dict__`), speed `tuple` jitni,
magar readability class jaisi.

```python
Point = namedtuple("Point", ["x", "y"])
p = Point(3, 4)
p.x, p[0]       # dono kaam do are
```

## why?
- `row[2]` of jagah `row.price` - code self-documenting
- Immutable -> hashable -> `dict` key / `set` member ban can is
- Function from multiple values return to do of clean tareeqa
- CSV / DB rows then wrap to do of for perfect (memory-friendly)

## Ahem API (sab undelayscore from shuru - field names from clash no is)
| Member | Kaam |
|---|---|
| `_fields` | field names of tuple |
| `_field_defaults` | defaults of dict |
| `_replace(**kw)` | **new** object with badli happened values (original safe) |
| `_asdict()` | `dict` me convert (JSON of for) |
| `_make(iterable)` | iterable from instance (CSV row -> record) |
| `defaults=` | rightmost fields of defaults (constructor arg) |

## namedtuple vs dataclass vs dict (interview favourite)
| | `namedtuple` | `dataclass` | `dict` |
|---|---|---|---|
| Mutable | not | yes (default) | yes |
| Memory | Sabse kam | Zyada | Sabse zyada |
| Hashable | yes | `frozen=True` from | not |
| Type hints | `NamedTuple` class from | Built-in | not |
| Unpacking / iteration | yes | not | keys but |

In [ ]:
# ------------------------------------------------------------------
# namedtuple: poora tour
# ------------------------------------------------------------------
# defaults= rightmost fields but lagta is (yahan: qty=1, discount=0.0)
Product = namedtuple("Product", ["name", "price", "qty", "discount"], defaults=(1, 0.0))

p1 = Product("Laptop", 250000, 2, 0.1)
p2 = Product("Mouse", 1500)                  # qty=1, discount=0.0 default from
print("p1 :", p1)
print("p2 :", p2)
print()

print("_fields          :", Product._fields)
print("_field_defaults  :", Product._field_defaults)
print("name from access   :", p1.name, "| index from:", p1[0], "| dono same:", p1.name == p1[0])
print("unpacking        :", end=" ")
name, price, qty, disc = p1
print(name, price, qty, disc)
print()

# _replace -> new object banata is (tuple immutable is, mutate not is)
p3 = p1._replace(qty=5)
print("_replace(qty=5)  :", p3)
print("original safe    :", p1)
assert p1.qty == 2 and p3.qty == 5 and p1 is not p3

# Immutability proof (deliberate error)
try:
    p1.price = 1
except Attributeerror as e:
    print("mutate error     :", e, " <-- immutable is, _replace use do")
print()

# _asdict -> JSON friendly
print("_asdict()        :", p1._asdict())
print("json             :", json.dumps(p1._asdict()))

# _make -> CSV / DB row from seedha record
raw_rows = [("Keyboard", 4500, 3, 0.05), ("Monitor", 42000, 1, 0.0)]
records = [Product._make(r) for r in raw_rows]
for r in records:
    total = r.price * r.qty * (1 - r.discount)
    print(f"  {r.name:<10} qty={r.qty} total={total:,.2f}")
print()

# Hashable -> set / dict key ban can is (dict not ban can!)
Point = namedtuple("Point", "x y")           # space-separated string bhi chalti is
pts = {Point(0, 0), Point(1, 2), Point(0, 0)}
print("set of points    :", sorted(pts))     # duplicate khud khatam
print("dict key         :", {Point(1, 1): "origin-ish"})
assert len(pts) == 2

# GOTCHA: namedtuple == plain tuple compare does is (naam ignore is is)
print()
print("Point(1,2) == (1,2) :", Point(1, 2) == (1, 2), " <-- True! tuple hi is")
Other = namedtuple("Other", "x y")
print("Point(1,2) == Other(1,2) :", Point(1, 2) == Other(1, 2), " <-- class farq not parta")

# Typed version (behtar - editors autocomplete dete are)
from typing import NamedTuple
class City(NamedTuple):
    name: str
    population: int
    province: str = "Punjab"      # default

khi = City("Karachi", 15_000_000, "Sindh")
lhr = City("Lahore", 11_000_000)
print()
print("typed NamedTuple :", khi, "|", lhr)
assert lhr.province == "Punjab"
assert isinstance(khi, tuple)

---

# A5. `OrdelayedDict` (ab kab useful?) and `ChainMap` (config layering)

## `OrdelayedDict` - Python 3.7 of after bhi zaroorat?
Python 3.7+ me **plain `dict` insertion ordelay guarantee does is** (3.6 me
implementation detail thi). then phir `OrdelayedDict` why? **4 asli wajhein:**

1. **`move_to_end(key, last=True/False)`** - `dict` me this not is. LRU cache of dil.
2. **`popitem(last=False)`** - `dict` of `popitem()` sirf last from nikalta is; `OrdelayedDict`
   dono taraf from nikal can is (queue-like behaviour).
3. **Ordelay-sensitive equality** - `OrdelayedDict` of `==` ordelay dekhta is, `dict` of not.
   this testing me kaam aata is jahan ordelay matter does is.
4. **Explicit intent / API contract** - code parhne those then pata chal jata is of
   "ordelay yahan matter does is, isse no chhero". Plus `reversed()` `dict` but 3.8+ runs is
   magar `OrdelayedDict` but hamesha runs tha.

> Interview answer: *"Plain dict ordelayed is, is for zyadatar OrdelayedDict of zaroorat khatam.
> Magar `move_to_end`, `popitem(last=False)`, and ordelay-sensitive equality of for ab bhi
> OrdelayedDict use does hoon - like LRU cache implement do time."*

---

## `ChainMap` - kai dictionaries of ek "view"

## what is?
`ChainMap(d1, d2, d3)` in dicts then **copy not does**. Lookup first `d1` me,
no mile then `d2` me, phir `d3` me. Pehla match jeetta is.

## why?
**Config layering** of textbook use-case:
```
CLI args   (sabse zyada priority)
  -> environment variables
    -> config file
      -> built-in defaults   (sabse kam priority)
```
`{**defaults, **file, **env, **cli}` bhi kaam does is, magar woh **copy** banata is.
`ChainMap`:
- Zero-copy (bara config is then memory bachti is)
- **Live** is - undelaylying dict badlo then `ChainMap` me nazar aa jayega
- `.maps` list from layers dekh/badal can is
- `.new_child()` from temporary scope (like nested function scopes / template context)

## Ahem gotcha
**Writes sirf first map (`maps[0]`) but go are.** `del` bhi sirf `maps[0]` from is is -
behind that layer me key is then `Keyerror` milega. `parents` property `maps[1:]` deti is.

In [ ]:
# ------------------------------------------------------------------
# OrdelayedDict: aaj bhi kaam of cheezein
# ------------------------------------------------------------------
od = OrdelayedDict([("a", 1), ("b", 2), ("c", 3)])
print("start            :", od)

od.move_to_end("a")                    # 'a' then aakhir me bhejo
print("move_to_end('a') :", od)
od.move_to_end("c", last=False)        # 'c' then shuru me bhejo
print("move_to_end('c', last=False):", od)

print("popitem()        :", od.popitem(), "-> ", od)                # last from
print("popitem(last=False):", od.popitem(last=False), "-> ", od)    # first from
print()

# Ordelay-sensitive equality: yahi asli farq is
d_a, d_b = {"x": 1, "y": 2}, {"y": 2, "x": 1}
o_a, o_b = OrdelayedDict(d_a), OrdelayedDict(d_b)
print("dict        {x,y} == {y,x} :", d_a == d_b, " <-- ordelay ignore")
print("OrdelayedDict {x,y} == {y,x} :", o_a == o_b, " <-- ordelay matters")
assert d_a == d_b
assert o_a != o_b
# NOTE: OrdelayedDict == plain dict compare ordelay-insensitive rehta is
print("OrdelayedDict == dict        :", o_a == d_b, " <-- mixed compare = ordelay ignore")
print()

# Asli use-case: LRU cache (move_to_end + popitem(last=False))
class LRUCache:
    '''Chhota LRU cache. get/put dono O(1). (LeetCode 146 of core.)'''
    def __init__(self, capacity: int):
        self.cap = capacity
        self.data = OrdelayedDict()          # front = sabse old, back = sabse new

    def get(self, key):
        if key not in self.data:
            return -1
        self.data.move_to_end(key)         # touch -> recent bana do
        return self.data[key]

    def put(self, key, value):
        if key in self.data:
            self.data.move_to_end(key)
        self.data[key] = value
        if len(self.data) > self.cap:
            evicted, _ = self.data.popitem(last=False)   # sabse old nikalo
            print(f"   (evicted: {evicted})")

cache = LRUCache(2)
cache.put("a", 1); cache.put("b", 2)
print("get a :", cache.get("a"))       # 'a' recent ban gaya
cache.put("c", 3)                      # capacity full -> 'b' evict hoga (a recent is)
print("keys  :", list(cache.data.keys()))
print("get b :", cache.get("b"))       # -1 (evicted)
assert cache.get("b") == -1 and cache.get("a") == 1

In [ ]:
# ------------------------------------------------------------------
# ChainMap: config layering (zero-copy, live)
# ------------------------------------------------------------------
defaults    = {"host": "localhost", "port": 8000, "debug": False, "workers": 1}
config_file = {"host": "db.bizee.internal", "port": 5432}
env_vars    = {"debug": True}
cli_args    = {"port": 9999}

# Priority: cli > env > file > defaults  (pehla map jeetta is)
cfg = ChainMap(cli_args, env_vars, config_file, defaults)

print("host    :", cfg["host"],    "(config_file from)")
print("port    :", cfg["port"],    "(cli_args from - sabse ahead)")
print("debug   :", cfg["debug"],   "(env_vars from)")
print("workers :", cfg["workers"], "(defaults from)")
print("as dict :", dict(cfg))
print("layers  :", len(cfg.maps), "->", cfg.maps)
print()

# LIVE is: undelaylying dict badlo, ChainMap me foran nazar aayega
config_file["host"] = "prod.bizee.internal"
print("file badla, cfg['host'] ab :", cfg["host"], " <-- copy not, view is")

# {**a, **b} of farq: woh SNAPSHOT is
merged = {**defaults, **config_file, **env_vars, **cli_args}
config_file["host"] = "another.host"
print("merged snapshot host       :", merged["host"], " <-- old (frozen)")
print("chainmap live host         :", cfg["host"])
print()

# WRITES sirf maps[0] but go are
cfg["timeout"] = 30
print("cli_args ab               :", cli_args, " <-- timeout yahan gaya")
assert "timeout" in cli_args and "timeout" not in defaults

# GOTCHA: del sirf maps[0] from is is
try:
    del cfg["workers"]                  # 'workers' defaults me is, maps[0] me not
except Keyerror as e:
    print("del error                 :", e)
    print("Fix                       : del cfg.maps[3]['workers']  (layer target do)")
print()

# new_child() -> temporary scope (function-call scope like)
base = ChainMap({"lang": "en"}, {"theme": "dark"})
temp = base.new_child({"lang": "ur"})       # new khaali/diya happened layer front me
print("base['lang'] :", base["lang"], "| temp['lang'] :", temp["lang"])
print("temp.parents == base layers :", temp.parents.maps == base.maps)
assert temp["theme"] == "dark" and base["lang"] == "en"

# Practical: nested template/variable scopes
scopes = ChainMap({}, {"user": "guest"}, {"user": "anon", "role": "readelay"})
scopes["user"] = "imran"                    # innermost scope me likha
print("resolved     :", dict(scopes))
assert scopes["user"] == "imran" and scopes["role"] == "readelay"

---

# B. `heapq` - priority queue / withoutry heap

## what is?
`heapq` ek **min-heap** is jo **plain `list` but** kaam does is (koi alag class not).
Invariant: `heap[k] <= heap[2*k+1]` and `heap[k] <= heap[2*k+2]`.
Matlab **`heap[0]` hamesha sabse chhota element is** - O(1) me.

> **Bara gotcha:** heap **sorted list not is**. Sirf root minimum guaranteed is.
> `print(heap)` do then inside of ordelay "random" lagega - this normal is.

## Complexity
| Operation | Time |
|---|---|
| `heap[0]` (peek min) | **O(1)** |
| `heappush(h, x)` | O(log n) |
| `heappop(h)` | O(log n) |
| `heapify(list)` | **O(n)** (sorting O(n log n) from fast!) |
| `heapreplace(h, x)` | O(log n) - **ek** sift, do from sasta |
| `nlargest(k, it)` / `nsmallest(k, it)` | O(n log k) |

## Kab heap, kab sorting?
- Poora ordelay should -> `sorted()`
- Sirf **top-k** ya **repeatedly minimum nikalna** -> heap
- Stream aa rahi is (data khatam not is) -> heap of size k

## why / kahan (real world)
- Task scheduler (next-deadline-first)
- Dijkstra / A* shortest path
- Merge K sorted streams (`heapq.merge`)
- Top-K trending items
- Median maintain to do (two heaps)

In [ ]:
# ------------------------------------------------------------------
# heapq basics: heappush / heappop / heapify / peek
# ------------------------------------------------------------------
h = []
for x in [5, 1, 8, 3, 9, 2]:
    heapq.heappush(h, x)
    print(f"push {x} -> {h}   (heap[0] = {h[0]})")
print()
print("Dhyan do: list SORTED not is, sirf h[0] minimum is.")
print("is_sorted?", h == sorted(h))
print()

print("pops (sorted ordelay me nikalte are):", end=" ")
out = []
while h:
    out.append(heapq.heappop(h))
print(out)
assert out == sorted([5, 1, 8, 3, 9, 2])
print()

# heapify: MOJOODA list then in-place heap banata is - O(n), sorting from fast
data = [9, 4, 7, 1, 8, 2, 6]
heapq.heapify(data)                 # in-place, return None
print("heapify of after :", data, "| min =", data[0])
assert data[0] == 1

# Heapsort (naive) - sirf samajhne of for; asal me sorted() use do
def heapsort(iterable):
    h = list(iterable)
    heapq.heapify(h)                          # O(n)
    return [heapq.heappop(h) for _ in range(len(h))]   # n x O(log n)

print("heapsort        :", heapsort([3, 1, 4, 1, 5, 9, 2, 6]))
assert heapsort([3, 1, 4, 1, 5, 9, 2, 6]) == [1, 1, 2, 3, 4, 5, 6, 9]
print()

# Khaali heap from pop -> Indexerror (deliberate error demo)
try:
    heapq.heappop([])
except Indexerror as e:
    print("empty pop error :", type(e).__name__, "->", e)
    print("Fix             : 'if heap:' check do ya try/except lagao")

In [ ]:
# ------------------------------------------------------------------
# nlargest / nsmallest / heapreplace / heappushpop
# ------------------------------------------------------------------
scores = [88, 45, 92, 67, 100, 23, 76, 95, 51]

print("nlargest(3)   :", heapq.nlargest(3, scores))
print("nsmallest(3)  :", heapq.nsmallest(3, scores))
print()

# key= support - dict/object list but direct runs is
employees = [
    {"name": "Ali",   "salary": 120000},
    {"name": "Sana",  "salary": 180000},
    {"name": "Bilal", "salary": 95000},
    {"name": "Hina",  "salary": 210000},
]
top2 = heapq.nlargest(2, employees, key=lambda e: e["salary"])
print("top 2 salaries:", [(e["name"], e["salary"]) for e in top2])
assert [e["name"] for e in top2] == ["Hina", "Sana"]
print()
print("Rule of thumb:")
print("  k == 1        -> min()/max() use do (fast tareen)")
print("  k ~ n         -> sorted()[:k] use do")
print("  1 << k << n   -> nlargest/nsmallest (O(n log k))")
print()

# ---- heapreplace: POP first, PUSH after me (size same rehta is) ----
hr = [1, 3, 5, 7]
heapq.heapify(hr)
popped = heapq.heapreplace(hr, 4)      # 1 nikla, 4 inside aaya
print("heapreplace(h, 4) -> popped:", popped, "| heap:", sorted(hr))
assert popped == 1

# ---- heappushpop: PUSH first, POP after me ----
hp = [1, 3, 5, 7]
heapq.heapify(hp)
popped2 = heapq.heappushpop(hp, 0)     # 0 push happened, phir min pop -> 0 hi wapas
print("heappushpop(h, 0) -> popped:", popped2, "| heap:", sorted(hp))
assert popped2 == 0                    # new element hi sabse chhota tha
print()
print("Farq remember:")
print("  heapreplace  = pop THEN push  -> new element us round me pop not is can")
print("  heappushpop  = push THEN pop  -> new element wapas aa can is")
print("  heapreplace khaali heap but Indexerror gives is, heappushpop not.")
print()

# 'top k of a stream' pattern - heap size k from zyada not barhne dete (memory O(k))
def top_k_stream(stream, k):
    h = []                                  # min-heap of size k
    for x in stream:
        if len(h) < k:
            heapq.heappush(h, x)
        elif x > h[0]:                      # new element sabse chhote from bara is?
            heapq.heapreplace(h, x)         # chhota nikalo, new daalo (1 sift)
    return sorted(h, reverse=True)

print("top 3 of stream:", top_k_stream(scores, 3))
assert top_k_stream(scores, 3) == [100, 95, 92]

# heapq.merge - kai sorted iterables then lazily merge does is (memory friendly)
a, b, c = [1, 4, 7], [2, 5, 8], [3, 6, 9]
print("heapq.merge    :", list(heapq.merge(a, b, c)))

## B2. Min-heap then max-heap banana + Priority Queue

### `heapq` sirf min-heap is. Max-heap should then?
**Trick: values negate kar do.** `-x` of minimum = `x` of maximum.

```python
h = [-x for x in nums]
heapq.heapify(h)
largest = -heapq.heappop(h)      # wapas nikaalte time phir negate
```
- Sirf **numbers** of for runs is (string negate not is).
- Strings/objects of for `key`-based wrapper class ya `nlargest` use do.
- `_heapify_max` / `_heappop_max` **private** are - production me no use do.

### Priority Queue tuples from
`heapq` tuples then **lexicographically** compare does is: first `[0]`, tie is then `[1]`, ahead.

```python
heapq.heappush(pq, (priority, task))
```

**Do bare gotchas:**
1. **Tie but `task` compare hoga.** if task `dict` ya custom object is then
   `Typeerror: '<' not supported` aayega. **Fix:** ek monotonically increasing
   `counter` beech me daalo: `(priority, counter, task)`. this FIFO tie-break bhi give gives is (stable).
2. **Priority `0` ya negative** exactly theek is - `heapq` sirf compare does is.

Interview me `queue.PriorityQueue` of zikar bhi do: woh thread-safe wrapper is
`heapq` of oopar, magar locking of wajah from single-thread me `heapq` fast is.

In [ ]:
# ------------------------------------------------------------------
# Max-heap trick + Priority Queue with tie-breaking counter
# ------------------------------------------------------------------
nums = [5, 1, 8, 3, 9, 2]

# MAX-HEAP via negation
mx = [-x for x in nums]
heapq.heapify(mx)
print("max-heap pops :", [-heapq.heappop(mx) for _ in range(len(mx))])
assert sorted(nums, reverse=True) == [9, 8, 5, 3, 2, 1]
print()

# ------------------------------------------------------------------
# GOTCHA: tuple tie but second element compare is is
# ------------------------------------------------------------------
pq_bad = []
heapq.heappush(pq_bad, (1, {"task": "email"}))
try:
    heapq.heappush(pq_bad, (1, {"task": "sms"}))   # same priority -> dict vs dict compare
except Typeerror as e:
    print("tie error     :", e)
    print("Fix           : (priority, counter, item) - counter from tie hamesha toot jata is")
print()

# ------------------------------------------------------------------
# Sahi Priority Queue: counter from stable FIFO tie-breaking
# ------------------------------------------------------------------
import itertools

class PriorityQueue:
    '''Min-heap based PQ. Chhoti priority = first nikle. Ties FIFO ordelay me.'''
    def __init__(self):
        self._heap = []
        self._counter = itertools.count()      # 0,1,2,... hamesha unique & increasing

    def push(self, item, priority):
        # counter beech me: priority tie is then insertion ordelay decision does is
        heapq.heappush(self._heap, (priority, next(self._counter), item))

    def pop(self):
        if not self._heap:
            raise Indexerror("pop from empty PriorityQueue")
        priority, _, item = heapq.heappop(self._heap)
        return item, priority

    def peek(self):
        return (self._heap[0][2], self._heap[0][0]) if self._heap else None

    def __len__(self):
        return len(self._heap)

pq = PriorityQueue()
pq.push({"task": "deploy"},        priority=2)
pq.push({"task": "fix-prod-bug"},  priority=1)
pq.push({"task": "write-tests"},   priority=2)   # 'deploy' of sath tie
pq.push({"task": "reply-email"},   priority=5)
pq.push({"task": "server-down"},   priority=0)   # sabse urgent

print("peek          :", pq.peek())
print("drain ordelay:")
ordelay = []
while len(pq):
    item, prio = pq.pop()
    ordelay.append(item["task"])
    print(f"  p={prio}  {item['task']}")

# 'deploy' 'write-tests' from first push happened tha -> tie me first nikla (stable)
assert ordelay == ["server-down", "fix-prod-bug", "deploy", "write-tests", "reply-email"]
print()

# Max-priority should (bara number = zyada urgent)? -> priority negate do
pq2 = PriorityQueue()
for task, importance in [("low", 1), ("critical", 10), ("medium", 5)]:
    pq2.push(task, -importance)              # negate = max-heap behaviour
print("max-priority ordelay:", [pq2.pop()[0] for _ in range(3)])

---

# C. `bisect` - sorted list but withoutry search (C speed)

## what is?
`bisect` ek **already sorted** list me withoutry search does is: O(log n).
this khud sort not does - **tumhari zimmedari is list sorted rakhna**.

## Two bunyadi functions
| Function | what gives is | Duplicates of sath |
|---|---|---|
| `bisect_left(a, x)` | pehla index jahan `x` daala ja can is | `x` of **barabar walon from first** |
| `bisect_right(a, x)` / `bisect(a, x)` | aakhri aisa index | `x` of barabar walon of **after** |

`a = [1, 2, 2, 2, 5]`, `x = 2`:
- `bisect_left  -> 1` (first 2 of jagah)
- `bisect_right -> 4` (aakhri 2 of after)
- `bisect_right - bisect_left = 3` = **`x` of count** (O(log n) me!)

## Insert helpers
- `insort_left(a, x)` / `insort_right(a, x)` = search **+** `a.insert(...)`
- **Search O(log n), magar insert O(n)** (elements shift are are) -> total O(n).
  Bohat inserts is then `list.append` + ek dafa `sort()`, ya `SortedList` (third-party) think.

## `key=` support (Python 3.10+)
`bisect_left(records, target, key=lambda r: r.score)` - list `key` of hisaab from sorted is.

## Kahan use is is
- **Grade / tax-slab / shipping-rate lookup** (range mapping, `if/elif` chain of jagah)
- Leadelayboard rank nikalna
- Sorted timestamps me "is time from first kitne events" (prefix count)
- LIS (Longest Increasing Subsequence) O(n log n)
- Weighted random selection (cumulative weights)

In [ ]:
# ------------------------------------------------------------------
# bisect: left vs right, insort, and classic lookups
# ------------------------------------------------------------------
a = [1, 2, 2, 2, 5, 8]
print("list          :", a)
print("bisect_left (a, 2) :", bisect.bisect_left(a, 2),  " <- first 2 of jagah")
print("bisect_right(a, 2) :", bisect.bisect_right(a, 2), " <- aakhri 2 of after")
print("count of 2         :", bisect.bisect_right(a, 2) - bisect.bisect_left(a, 2))
assert bisect.bisect_left(a, 2) == 1
assert bisect.bisect_right(a, 2) == 4
assert bisect.bisect_right(a, 2) - bisect.bisect_left(a, 2) == a.count(2)

# Maujood no hone that value: dono same index dete are (insert position)
print("bisect_left (a, 4) :", bisect.bisect_left(a, 4), "| bisect_right:", bisect.bisect_right(a, 4))
print()

# ---- 'exists?' check bisect from (withoutry search) ----
def contains(sorted_list, x) -> bool:
    i = bisect.bisect_left(sorted_list, x)
    return i < len(sorted_list) and sorted_list[i] == x

print("contains(a, 5) :", contains(a, 5), "| contains(a, 4):", contains(a, 4))
assert contains(a, 8) and not contains(a, 7)
print()

# ---- insort: sorted list maintain to do ----
sorted_scores = []
for s in [55, 91, 72, 12, 88, 72]:
    bisect.insort(sorted_scores, s)          # = insort_right
    print(f"insort {s:>2} -> {sorted_scores}")
assert sorted_scores == sorted(sorted_scores)
print()

# ---- GRADE LOOKUP: if/elif chain of jagah bisect ----
# breakpoints: 60, 70, 80, 90   grades: F  D  C  B  A
BREAKS = [60, 70, 80, 90]
GRADES = "FDCBA"

def grade(score: int) -> str:
    # bisect_right: 90 -> index 4 -> 'A' (yani 90 bhi A is, inclusive lower bound)
    return GRADES[bisect.bisect_right(BREAKS, score)]

for s in [0, 59, 60, 69, 70, 79, 85, 89, 90, 100]:
    print(f"  score {s:>3} -> {grade(s)}")
assert grade(59) == "F" and grade(60) == "D" and grade(90) == "A" and grade(100) == "A"
print()

# ---- TAX SLAB (Pakistan-style illustrative slabs) ----
SLAB_LIMITS = [600_000, 1_200_000, 2_400_000, 3_600_000]
SLAB_RATES  = [0.0,     0.025,     0.125,     0.20,      0.25]

def tax_rate(income: int) -> float:
    return SLAB_RATES[bisect.bisect_right(SLAB_LIMITS, income)]

for inc in [500_000, 600_000, 900_000, 2_000_000, 5_000_000]:
    print(f"  income {inc:>9,} -> rate {tax_rate(inc):.1%}")
assert tax_rate(500_000) == 0.0 and tax_rate(5_000_000) == 0.25
print()

# ---- key= (Python 3.10+): objects but bisect ----
Rec = namedtuple("Rec", "name score")
recs = [Rec("Ali", 40), Rec("Sana", 65), Rec("Bilal", 80), Rec("Hina", 95)]   # score from sorted
idx = bisect.bisect_left(recs, 70, key=lambda r: r.score)
print("pehla record with score >= 70 :", recs[idx])
assert recs[idx].name == "Bilal"
print()

# ---- GOTCHA: unsorted list but bisect = bakwaas jawab (koi error not) ----
bad = [5, 1, 9, 3]
print("unsorted but bisect_left(bad, 3):", bisect.bisect_left(bad, 3), " <-- silently GHALAT")
print("Sabaq: bisect use to do from first list sorted honi should.")

---

# D. `re` - Regular Expressions (deep dive)

## what is?
Regex ek chhoti si **pattern language** is text search/validate/replace of for.
Python me `re` module. Engine **backtracking** based is (NFA) - is for taqatwar is
magar khatarnaak bhi (neeche "catastrophic backtracking" parhna).

## Pehla and sabse ahem rule: **raw strings use do**
```python
re.search("\\d+", s)     # bad: Python first "\\d" then process does is
re.search(r"\d+", s)     # SAHI: r"" me backslash like is waisa engine tak jata is
```
`"\b"` Python me **backspace character** (0x08) is, regex of word-boundary not.
`r"\b"` sahi is. **Hamesha `r""` write** - koi exception not.

---

## D1. Literals and character classes

| Pattern | Matlab |
|---|---|
| `abc` | literal "abc" |
| `.` | koi bhi ek character **siwa newline** (`DOTALL` from newline bhi) |
| `[aeiou]` | in me from koi ek (character class / set) |
| `[a-z]` | range: a from z |
| `[a-zA-Z0-9_]` | multiple ranges jama |
| `[^abc]` | **negated**: a, b, c of ilawa some bhi |
| `[.]` ya `\.` | literal dot (class of inside `.` special not) |
| `a\|b` | alternation: `a` ya `b` |

**Class of inside of rules (log yahan galti do are):**
- inside `.` `*` `+` `(` `)` `?` special **not** are: `[.*+]` teen literal chars are.
- `-` then literal should then shuru/aakhir me rakho (`[-a-z]` ya `[a-z-]`) ya escape (`[a\-z]`).
- `^` sirf **pehli position** but negation is: `[a^b]` me `^` literal is.
- `]` then literal should then first rakho ya escape: `[]a]` / `[\]a]`.

## D2. Predefined (shorthand) classes

| Shorthand | Barabar | Negation |
|---|---|---|
| `\d` | `[0-9]` (+ Unicode digits!) | `\D` = non-digit |
| `\w` | `[a-zA-Z0-9_]` (+ Unicode letters) | `\W` = non-word |
| `\s` | `[ \t\n\r\f\v]` | `\S` = non-space |

**Unicode gotcha:** default me `str` patterns **Unicode-aware** are are. `\d` Arabic-Indic
digits (`٣`) and `\w` Urdu/Chinese letters bhi match does is. Sirf ASCII should then
`re.ASCII` flag do ya `[0-9]` / `[a-zA-Z0-9_]` explicit write.

In [ ]:
# ------------------------------------------------------------------
# Literals, character classes, predefined classes
# ------------------------------------------------------------------
s = "Ordelay #A-42 shipped then Lahore on 2024-03-15 for Rs. 15,000."

print("literal 'Lahore' :", re.search(r"Lahore", s))          # Match object ya None
print("dot '.' (any)    :", re.findall(r"L.h", s))            # L + koi 1 char + h
print("vowels           :", re.findall(r"[aeiou]", s)[:8], "...")
print("digit runs       :", re.findall(r"[0-9]+", s))
print("NOT digit/space  :", re.findall(r"[^0-9\s]+", s)[:5], "...")
print()

# Class of inside special chars literal is go are
print(r"[.*+] literal    :", re.findall(r"[.*+]", "a.b*c+d"))
print(r"escaped \. dot   :", re.findall(r"\.", s))
print(r"[-+] dash first  :", re.findall(r"[-+]", "a-b+c"))     # '-' shuru me = literal
print(r"[a^b] caret mid  :", re.findall(r"[a^b]", "a^b"))      # '^' beech me = literal
print()

# --- Predefined classes ---
line = "Ali_123 scored 87% on 2024-05-01\tGrade: A+"
print(r"\d+  :", re.findall(r"\d+", line))
print(r"\D+  :", re.findall(r"\D+", line)[:3], "...")
print(r"\w+  :", re.findall(r"\w+", line))
print(r"\W+  :", re.findall(r"\W+", line))
print(r"\s+  :", re.findall(r"\s+", line))       # space and tab
print(r"\S+  :", re.findall(r"\S+", line))
print()

# --- UNICODE gotcha ---
mixed = "abc ٣٤ 123 اردو"       # arabic digits + urdu word
print("unicode-aware \\d :", re.findall(r"\d+", mixed))        # arabic digits BHI aaye
print("ASCII flag    \\d :", re.findall(r"\d+", mixed, re.ASCII))
print("unicode-aware \\w :", re.findall(r"\w+", mixed))        # urdu word bhi \w is
print("ASCII flag    \\w :", re.findall(r"\w+", mixed, re.ASCII))
assert re.findall(r"\d+", mixed, re.ASCII) == ["123"]
print()

# --- re.escape: user input then literal banane of SAHI tareeqa ---
user_typed = "price (Rs.) +tax?"
print("raw pattern from search to do crash/ghalat give can is; re.escape use do:")
print("escaped :", re.escape(user_typed))
print("found   :", re.search(re.escape(user_typed), f"Column: {user_typed} here") is not None)
try:
    re.compile(user_typed)                # unbalanced paren is then error
except re.error as e:
    print("re.error:", e)

## D3. Quantifiers - greedy vs lazy vs possessive

| Quantifier | Matlab |
|---|---|
| `*` | 0 ya zyada |
| `+` | 1 ya zyada |
| `?` | 0 ya 1 (optional) |
| `{n}` | exactly n |
| `{n,}` | n ya zyada |
| `{,m}` | 0 from m tak |
| `{n,m}` | n from m tak |

Har quantifier of **teen mode** are:

| Mode | Likhne of tareeqa | Behaviour | Backtrack? |
|---|---|---|---|
| **Greedy** (default) | `.*` `a+` `\d{2,5}` | **max** lo, phir zaroorat but behind hato | yes |
| **Lazy / non-greedy** | `.*?` `a+?` `\d{2,5}?` | **min** lo, phir zaroorat but ahead barho | yes |
| **Possessive** (3.11+) | `.*+` `a++` `\d{2,5}+` | max lo and **kabhi behind no hato** | **not** |

Possessive of bhai: **atomic group** `(?>...)` (bhi 3.11+). Dono backtracking band kar
dete are -> catastrophic backtracking from bachao, and fast.

### Classic example
Text: `<b>bold</b> and <i>italic</i>`
- `<.*>`  -> **poora** `<b>bold</b> and <i>italic</i>` (greedy!)
- `<.*?>` -> `<b>` (lazy - sirf pehla tag)
- `<[^>]*>` -> `<b>` (**behtar tareeqa** - lazy from bhi fast, because backtracking not)

> **Interview line:** *"Greedy `.*` first poora line kha jata is, phir behind hat kar
> match banata is. Lazy `.*?` ulta does is. Magar best practice negated class
> (`[^>]*`) is because woh deterministic is - backtracking hi not is."*

---

## D4. Anchors - position match do are, character not

| Anchor | Matlab |
|---|---|
| `^` | string of shuru (`MULTILINE` me har line of shuru) |
| `$` | string of aakhir, **ya aakhri `\n` from first** (`MULTILINE` me har line of aakhir) |
| `\A` | string of shuru - `MULTILINE` from **kabhi** affect not is |
| `\Z` | string of exactly aakhir - trailing newline of koi rehmat not |
| `\b` | word boundary (`\w` and non-`\w` of darmiyan) |
| `\B` | **non**-boundary (word of inside) |

**`$` of bara gotcha:** `re.search(r"abc$", "abc\n")` **match does is!**
exactly strict should then `\Z` use do.

Anchors **zero-width** are - koi character consume not do.

In [ ]:
# ------------------------------------------------------------------
# Greedy vs Lazy vs Possessive
# ------------------------------------------------------------------
html = "<b>bold</b> and <i>italic</i>"

print("GREEDY <.*>     :", re.findall(r"<.*>", html))
print("LAZY   <.*?>    :", re.findall(r"<.*?>", html))
print("BEST   <[^>]*>  :", re.findall(r"<[^>]*>", html))
assert re.findall(r"<.*>", html) == [html]                       # poora string!
assert re.findall(r"<.*?>", html) == ["<b>", "</b>", "<i>", "</i>"]
print()

# Number example: greedy behind hat kar match banata is
num = "12345"
print("greedy \\d{2,4}   :", re.match(r"\d{2,4}", num).group())       # '1234' (max)
print("lazy   \\d{2,4}?  :", re.match(r"\d{2,4}?", num).group())      # '12'   (min)
print()

# Greedy backtracking of nazara: \d+ then aakhri digit chhorni parti is
m = re.match(r"(\d+)(\d)", "12345")
print(r"(\d+)(\d) on 12345 -> group1 =", m.group(1), ", group2 =", m.group(2))
print("   \\d+ ne first 12345 liya, phir behind hat kar 1234 but aaya (backtrack)")
print()

# POSSESSIVE / ATOMIC (Python 3.11+) - backtrack to do from INKAR
try:
    print(r"possessive (\d++)(\d) :", re.match(r"(\d++)(\d)", "12345"))
    print("   None got -> \\d++ ne 12345 poora le liya and behind not hata,")
    print("   is for doosre (\\d) of for some no bacha -> fail.")
    print(r"atomic     (?>\d+)\d  :", re.match(r"(?>\d+)\d", "12345"))
    print(r"possessive .*+ alone  :", re.fullmatch(r"\d*+", "999"))
except re.error as e:
    print("Possessive/atomic support not (Python < 3.11):", e)
print()

# {n,m} of saare roop
for pat in [r"^a{3}$", r"^a{2,}$", r"^a{,2}$", r"^a{2,4}$"]:
    results = {t: bool(re.match(pat, t)) for t in ["a", "aa", "aaa", "aaaaa"]}
    print(f"  {pat:<10} -> {results}")
print()

# GOTCHA: '?' of do exactly alag kaam
print("optional '?'  colou?r :", re.findall(r"colou?r", "color and colour"))
print("lazy     '?'  o+?     :", re.match(r"o+?", "ooo").group(), "(sirf 1)")

In [ ]:
# ------------------------------------------------------------------
# Anchors: ^ $ \A \Z \b \B
# ------------------------------------------------------------------
text = "cat catalog concat the cat sat"

print("plain  'cat'  :", len(re.findall(r"cat", text)), "matches (catalog/concat bhi)")
print(r"\bcat\b      :", re.findall(r"\bcat\b", text), "<- poora word 'cat' sirf")
print(r"\bcat        :", re.findall(r"\bcat\w*", text), "<- 'cat' from SHURU hone those")
print(r"cat\b        :", re.findall(r"\w*cat\b", text), "<- 'cat' but KHATAM hone those")
print(r"\Bcat        :", re.findall(r"\Bcat\w*", text), "<- word of inside (boundary not)")
assert re.findall(r"\bcat\b", text) == ["cat", "cat"]
print()

# ^ and $ (single line)
print("^cat  :", bool(re.search(r"^cat", text)))
print("sat$  :", bool(re.search(r"sat$", text)))
print()

# ---- $ of bara GOTCHA: trailing newline ----
print(r"re.search(r'abc$',  'abc\n') :", bool(re.search(r"abc$", "abc\n")),  " <-- TRUE! surprise")
print(r"re.search(r'abc\Z', 'abc\n') :", bool(re.search(r"abc\Z", "abc\n")), " <-- strict")
print(r"re.fullmatch(r'abc','abc\n') :", bool(re.fullmatch(r"abc", "abc\n")))
assert re.search(r"abc$", "abc\n") is not None
assert re.search(r"abc\Z", "abc\n") is None
print()

# ---- MULTILINE me ^ $ per-line ban go are, \A \Z not ----
log = "line one\nline two\nline three"
print("^line  (no flag)  :", len(re.findall(r"^line", log)), "match")
print("^line  (MULTILINE):", len(re.findall(r"^line", log, re.M)), "matches")
print(r"\Aline (MULTILINE):", len(re.findall(r"\Aline", log, re.M)), "match  <-- flag of asar not")
assert len(re.findall(r"^line", log, re.M)) == 3
assert len(re.findall(r"\Aline", log, re.M)) == 1
print()

# \b of definition Unicode/undelayscore but surprise deti is
print(r"\bfoo\b in 'foo_bar' :", bool(re.search(r"\bfoo\b", "foo_bar")), " <-- False, '_' word-char is")
print(r"\bfoo\b in 'foo-bar' :", bool(re.search(r"\bfoo\b", "foo-bar")), " <-- True, '-' non-word")

# Practical: whole-word replace (naive replace from behtar)
sentence = "I like Java. Javascript is not Java."
print()
print("str.replace :", sentence.replace("Java", "Python"))       # Javascript bhi bigar gaya!
print("regex \\b    :", re.sub(r"\bJava\b", "Python", sentence))  # sahi
assert re.sub(r"\bJava\b", "Python", sentence) == "I like Python. Javascript is not Python."

## D5. Groups, non-capturing, named groups, alternation, backreferences, lookaround

### Groups `( ... )`
- Sub-pattern then **capture** do are -> `m.group(1)`, `m.groups()`
- Quantifier of scope banate are: `(ab)+` = "ab" repeat, `ab+` = "ab", "abb", ...
- `findall` **groups return does is**, poora match not - classic surprise!

### Non-capturing `(?: ... )`
Grouping should magar capture not -> fast, clean, and group numbers shift not are.
```python
r"(?:Mr|Mrs|Ms)\. (\w+)"      # title group not, naam group 1
```

### Named groups `(?P<name> ... )`
```python
m = re.match(r"(?P<y>\d{4})-(?P<m>\d{2})", "2024-03")
m.group("y"), m.groupdict()      # '2024', {'y': '2024', 'm': '03'}
```
Backreference: `(?P=name)` pattern me, `\g<name>` replacement me.
**Named groups log-parsing of for best are** - `group(7)` yaad rakhne from bachao.

### Alternation `|`
- **Left-then-right, pehla jo match kar jaye** (longest not!). `r"a|ab"` but "ab" -> `"a"` milega.
- Scope poore pattern but is is -> `(?:...)` from ghero: `r"^(?:cat|dog)$"`, no of `r"^cat|dog$"`.
- Zyada specific alternative first write: `r"\b(?:Javascript|Java)\b"`.

### Backreferences `\1`, `\2`, `(?P=name)`
first capture of happened **exact same text** dobara match do:
- `r"\b(\w+) \1\b"` -> duplicate word ("the the")
- `r"(['\"]).*?\1"` -> matching quotes
- `re.sub(r"(\w+)@(\w+)", r"\2 at \1", s)` -> replacement me `\1` `\2`

### Lookaround (zero-width assertions)
| Syntax | Naam | Matlab |
|---|---|---|
| `(?=...)` | positive lookahead | ahead this hona should (consume not) |
| `(?!...)` | negative lookahead | ahead this **no** is |
| `(?<=...)` | positive lookbehind | behind this hona should |
| `(?<!...)` | negative lookbehind | behind this **no** is |

**Python of lookbehind fixed-width honi should:** `(?<=abc)` theek, `(?<=a+)` **error**.
(Workaround: `(?<=aa|bb)` same length alternatives, ya text ulta do.)

Lookarounds "AND conditions" lagane of for best are - password strength of classic pattern.

In [ ]:
# ------------------------------------------------------------------
# Groups, non-capturing, named groups, alternation
# ------------------------------------------------------------------
m = re.match(r"(\d{4})-(\d{2})-(\d{2})", "2024-03-15")
print("group(0) poora  :", m.group(0))
print("group(1..3)     :", m.group(1), m.group(2), m.group(3))
print("groups()        :", m.groups())
print("group(1,3) tuple:", m.group(1, 3))
print("span(2)         :", m.span(2), "| start/end:", m.start(2), m.end(2))
print()

# findall of SURPRISE: groups hon then groups get are
print("findall no group :", re.findall(r"\d{4}-\d{2}-\d{2}", "2024-03-15 and 2025-01-01"))
print("findall 1 group  :", re.findall(r"(\d{4})-\d{2}-\d{2}", "2024-03-15 and 2025-01-01"))
print("findall 3 groups :", re.findall(r"(\d{4})-(\d{2})-(\d{2})", "2024-03-15 and 2025-01-01"))
print("Fix (poora match should) -> (?:...) non-capturing use do:")
print("  ", re.findall(r"(?:\d{4})-(?:\d{2})-(?:\d{2})", "2024-03-15 and 2025-01-01"))
print()

# Group + quantifier scope
print("(ab)+ on 'ababab' :", re.match(r"(ab)+", "ababab").group(0), "| last capture:", re.match(r"(ab)+", "ababab").group(1))
print("ab+   on 'abbb'   :", re.match(r"ab+", "abbb").group(0))
print()

# NAMED groups - readable code
pat = re.compile(r"(?P<user>[\w.]+)@(?P<domain>[\w.]+)\.(?P<tld>\w{2,})")
mm = pat.match("imran.bizee@cyggs.tech")
print("groupdict()     :", mm.groupdict())
print("group('domain') :", mm.group("domain"))
assert mm.group("user") == "imran.bizee" and mm.group("tld") == "tech"

# Optional group jo match no is -> None (empty string not)
mo = re.match(r"(\d+)(?:\.(\d+))?", "42")
print("optional group  :", mo.groups(), " <-- doosra None is")
print("groups(default) :", mo.groups(default="0"), " <-- default give can is")
assert mo.group(2) is None
print()

# ALTERNATION: pehla match jeetta is (longest not)
print(r"'a|ab'  on 'ab' :", re.match(r"a|ab", "ab").group(), " <-- 'a' (pehla alternative)")
print(r"'ab|a'  on 'ab' :", re.match(r"ab|a", "ab").group(), " <-- specific first write")
print(r"^cat|dog$ scope :", bool(re.match(r"^cat|dog$", "catdog")), " <-- galat scope!")
print(r"^(?:cat|dog)$   :", bool(re.match(r"^(?:cat|dog)$", "catdog")), " <-- sahi")
print()

# ------------------------------------------------------------------
# BACKREFERENCES
# ------------------------------------------------------------------
# 1) Duplicate word pakro
dup = "this is is a test test sentence with the the bug"
print("duplicate words :", re.findall(r"\b(\w+)\s+\1\b", dup))
print("fixed           :", re.sub(r"\b(\w+)\s+\1\b", r"\1", dup))
assert re.sub(r"\b(\w+)\s+\1\b", r"\1", dup) == "this is a test sentence with the bug"

# 2) Matching quotes (single ya double, magar dono same)
q = "He said \"hello\" and 'bye' but \"mismatched'"
print("quoted strings  :", re.findall(r"(['\"])(.*?)\1", q))

# 3) Named backreference (?P=name)
print("named backref   :", re.findall(r"(?P<c>\w)(?P=c)", "aabbcdee"))   # double letters

# 4) Repeated HTML-ish tag pair
print("tag pair        :", re.findall(r"<(\w+)>(.*?)</\1>", "<b>bold</b><i>it</i>"))

# 5) sub me \g<name> - jab \1 of after digit aata is
print("swap with \\g<>  :", re.sub(r"(?P<a>\w+)@(?P<b>\w+)", r"\g<b>_at_\g<a>", "ali@bizee"))
print("digit clash fix :", re.sub(r"(\d)", r"\g<1>0", "5"), "  <-- r'\\10' is then group 10 samjha jata")
assert re.sub(r"(\d)", r"\g<1>0", "5") == "50"

In [ ]:
# ------------------------------------------------------------------
# LOOKAHEAD / LOOKBEHIND (positive + negative)
# ------------------------------------------------------------------
prices = "Laptop Rs.250000, Mouse USD 15, Bag Rs.4500, Pen EUR 3"

# (?<=...) positive lookbehind: 'Rs.' of after those numbers (Rs. match me shamil not)
print("after Rs.  (?<=Rs\\.)\\d+ :", re.findall(r"(?<=Rs\.)\d+", prices))

# (?=...) positive lookahead: aisa word jiske after ' Rs.' aata is
print("before Rs. \\w+(?= Rs\\.) :", re.findall(r"\w+(?= Rs\.)", prices))

# (?<!...) negative lookbehind: numbers jinke behind 'Rs.' not is
print("not after Rs.           :", re.findall(r"(?<!Rs\.)\b\d+\b", prices))

# (?!...) negative lookahead: 'a' from shuru word jo 'and' no is
words = "apple and ant android answer"
print("a-word not 'and'        :", re.findall(r"\ba(?!nd\b)\w*", words))
print()

# --- Thousands separator daalna (classic lookahead puzzle) ---
n = "1234567890"
print("commas via lookahead :", re.sub(r"\B(?=(\d{3})+(?!\d))", ",", n))
assert re.sub(r"\B(?=(\d{3})+(?!\d))", ",", n) == "1,234,567,890"
print("   (?=(\\d{3})+(?!\\d)) = 'ahead from aakhir tak digits of poore 3-3 group banein'")
print("   \\B = string of shuru me comma no lage")
print()

# --- Multiple AND conditions ek pattern me (lookaheads zero-width are,")
#     is for sab same position from check are are) ---
strong = re.compile(r'''
    ^                 # shuru
    (?=.*[a-z])       # kahin ek lowercase is
    (?=.*[A-Z])       # kahin ek uppercase is
    (?=.*\d)          # kahin ek digit is
    (?=.*[^\w\s])     # kahin ek symbol is
    (?!.*\s)          # koi whitespace no is
    .{8,}             # kam az kam 8 chars
    $
''', re.VERBOSE)
for pw in ["Pak@1234", "weakpass", "NOLOWER1@", "Has Space1@", "Sh0rt@"]:
    print(f"  {pw!r:<16} -> {'STRONG' if strong.match(pw) else 'weak'}")
assert strong.match("Pak@1234") and not strong.match("weakpass")
print()

# --- Lookbehind FIXED-WIDTH hona should (Python limitation) ---
try:
    re.compile(r"(?<=a+)b")
except re.error as e:
    print("variable-width lookbehind error :", e)
    print("Workaround : (?<=aa|bb) same-length alternatives, ya \\K that engine (Python me not)")
print()

# --- Lookaround zero-width is: consume not does -> overlapping matches ---
print("overlapping via lookahead:", re.findall(r"(?=(\d\d))", "12345"))
print("   normal findall(r'\\d\\d')  :", re.findall(r"\d\d", "12345"), "<-- overlap not")

## D6. Flags - pattern of behaviour badalne those switches

| Flag | Short | Kaam |
|---|---|---|
| `re.IGNORECASE` | `re.I` | case-insensitive |
| `re.MULTILINE` | `re.M` | `^` `$` har line but |
| `re.DOTALL` | `re.S` | `.` newline bhi match kare |
| `re.VERBOSE` | `re.X` | pattern me whitespace + `#` comments allowed |
| `re.ASCII` | `re.A` | `\d \w \s \b` sirf ASCII |
| `re.UNICODE` | `re.U` | default for `str` (redundant) |
| `re.NOFLAG` | - | 0 (koi flag not, 3.11+) |

Combine `|` from: `re.I | re.M`.

**Inline flags** pattern of inside bhi likh can is:
- `(?i)` poore pattern but (Python me **pattern of shuru me** hona should, warna error)
- `(?i:...)` sirf us group but (**scoped**, 3.6+) - this zyada mehfooz is
- `(?-i:...)` us hisse but flag **off**

### `VERBOSE` (X-mode) - production regex of must
```python
pat = re.compile(r'''
    ^(?P<code>[A-Z]{3})     # 3 uppercase letters = airport code
    -(?P<num>\d{3,4})$      # dash + 3-4 digit flight number
''', re.VERBOSE)
```
VERBOSE me:
- **Un-escaped whitespace ignore is is** -> space should then `\ ` ya `[ ]` ya `\s` write
- `#` from line of aakhir tak comment
- Character class **of inside** whitespace ignore **not** is (`[a b]` me space literal is)

In [ ]:
# ------------------------------------------------------------------
# Flags: I, M, S, X + inline scoped flags
# ------------------------------------------------------------------
# IGNORECASE
print("no flag  :", re.findall(r"python", "Python PYTHON python PyThOn"))
print("re.I     :", re.findall(r"python", "Python PYTHON python PyThOn", re.I))
print()

# MULTILINE
csvish = "name,age\nAli,25\nSana,30\nBilal,28"
print("^\\w+ no flag :", re.findall(r"^\w+", csvish))
print("^\\w+ re.M    :", re.findall(r"^\w+", csvish, re.M))
print("\\w+$ re.M    :", re.findall(r"\w+$", csvish, re.M))
assert re.findall(r"^\w+", csvish, re.M) == ["name", "Ali", "Sana", "Bilal"]
print()

# DOTALL - '.' newline bhi khaye
block = "START\nline1\nline2\nEND"
print("START.*END no flag :", re.search(r"START.*END", block))
print("START.*END re.S    :", repr(re.search(r"START.*END", block, re.S).group()[:20]) + "...")
assert re.search(r"START.*END", block) is None
assert re.search(r"START.*END", block, re.S) is not None
print()

# Combining flags
print("re.I|re.M :", re.findall(r"^ali", "ALI\nali\nAli", re.I | re.M))
print()

# VERBOSE: commented, readable pattern
flight_re = re.compile(r'''
    ^                       # string of shuru
    (?P<origin>[A-Z]{3})    # 3-letter origin code, e.g. LHE
    \s*->\s*                # arrow, aas paas whitespace optional
    (?P<dest>[A-Z]{3})      # 3-letter destination code
    \s+                      # separator (escape/class zaroori not, \s is)
    (?P<flight>[A-Z]{2}\d{3,4})   # airline code + number, e.g. PK301
    $                       # aakhir
''', re.VERBOSE)

for line in ["LHE -> KHI PK301", "ISB->DXB EK615", "lhe -> khi pk301", "LHE KHI PK301"]:
    mt = flight_re.match(line)
    print(f"  {line!r:<22} -> {mt.groupdict() if mt else None}")
assert flight_re.match("LHE -> KHI PK301").group("flight") == "PK301"
print()

# VERBOSE gotcha: literal space should then escape do
print("VERBOSE me plain space ignore:", bool(re.match(r"a b", "ab", re.X)))
print("escaped space \\  kaam does  :", bool(re.match(r"a\ b", "a b", re.X)))
print("[ ] class bhi theek           :", bool(re.match(r"a[ ]b", "a b", re.X)))
assert re.match(r"a b", "ab", re.X) is not None
print()

# Inline flags
print("(?i) at start   :", re.findall(r"(?i)python", "Python PYTHON"))
print("(?i:..) scoped  :", re.findall(r"(?i:hello) world", "HELLO world | HELLO WORLD"))
try:
    re.compile(r"abc(?i)")          # global inline flag beech me = error (3.11+)
except re.error as e:
    print("inline flag mid-pattern error :", e)

## D7. `re` of poora API + compiled patterns + catastrophic backtracking

| Function | what gives is | Kahan match dhundta is |
|---|---|---|
| `re.match(p, s)` | `Match` ya `None` | **sirf shuru** from |
| `re.fullmatch(p, s)` | `Match` ya `None` | **poori** string match is |
| `re.search(p, s)` | `Match` ya `None` | **kahin bhi**, pehla |
| `re.findall(p, s)` | `list[str]` ya `list[tuple]` | saare non-overlapping |
| `re.finditer(p, s)` | **iterator of `Match`** | saare (memory-friendly, span/groupdict milta is) |
| `re.sub(p, repl, s)` | nayi string | replace all (`count=` from limit) |
| `re.subn(p, repl, s)` | `(new_string, n_replacements)` | jab count should |
| `re.split(p, s)` | `list[str]` | pattern but tod do (capture group is then separators bhi aate are) |

**Validation of for `fullmatch` use do, `match` not.** `re.match(r"\d+", "123abc")`
match kar jata is (sirf shuru dekhta is) - `fullmatch` not karega.

### `sub` of with **function** repl
```python
re.sub(r"\d+", lambda m: str(int(m.group()) * 2), "a1 b2")   # 'a2 b4'
```
Function then `Match` milta is, string return to do is is. Conditional/computed
replacements of for this bohat taqatwar is.

### Compiled patterns
`re.compile()` ek dafa parse does is, phir reuse. `re` module of inside bhi ~512
patterns of cache is, is for faida chhota is is - magar **loops me and bare patterns but
compile to do behtar is**, plus `re.VERBOSE` of sath readable code milta is.

### Catastrophic backtracking (ReDoS) - **this dont chalao**
```python
BAD  = r"^(a+)+$"          # nested quantifier
TEXT = "a" * 40 + "b"      # match fail hoga
# ^ engine 2**40 raaste try karega -> minute/ghante lag can are
```
Wajah: **nested quantifiers** (`(a+)+`, `(a*)*`, `(\s|\t)+$`) + fail hone that input.
Engine of paas "a" then groups me batne of exponentially bohat tareeqe are are.

**Ilaaj:**
1. Nested quantifier no write: `^a+$` kaafi is
2. Possessive/atomic: `^(?>a+)+$` ya `^(a++)+$`
3. Negated class from deterministic create: `"[^"]*"` no of `".*?"`
4. Ambiguous alternation `(a|a)*` from bacho
5. Input length limit do, and user-supplied regex **kabhi** trust no do
6. `regex` (third-party) library timeout support karti is; stdlib `re` me timeout **not** is

In [ ]:
# ------------------------------------------------------------------
# match vs fullmatch vs search vs findall vs finditer
# ------------------------------------------------------------------
s = "123abc456"

print("match(r'\\d+')     :", re.match(r"\d+", s))        # shuru from match -> '123'
print("fullmatch(r'\\d+') :", re.fullmatch(r"\d+", s))    # poora not -> None
print("search(r'[a-z]+') :", re.search(r"[a-z]+", s))     # kahin bhi -> 'abc'
print("match(r'[a-z]+')  :", re.match(r"[a-z]+", s))      # shuru me letter not -> None
print("findall(r'\\d+')   :", re.findall(r"\d+", s))
print()
print("VALIDATION of sabaq: match kaafi not!")
print("  match(r'\\d+', '123abc')     :", bool(re.match(r"\d+", "123abc")), " <-- ghalat 'valid'")
print("  fullmatch(r'\\d+', '123abc') :", bool(re.fullmatch(r"\d+", "123abc")), " <-- sahi")
assert re.match(r"\d+", "123abc") and not re.fullmatch(r"\d+", "123abc")
print()

# finditer -> Match objects (span, groupdict, lazy)
log = "error at 10:30, WARN at 11:45, error at 12:00"
for mt in re.finditer(r"(?P<level>error|WARN) at (?P<time>\d{2}:\d{2})", log):
    print(f"  span={mt.span()} level={mt.group('level'):<5} time={mt.group('time')}")
print()

# ------------------------------------------------------------------
# sub / subn / split
# ------------------------------------------------------------------
print("sub simple  :", re.sub(r"\s+", " ", "too    many\t\tspaces\n\nhere"))
print("sub count=1 :", re.sub(r"a", "X", "banana", count=1))
print("subn        :", re.subn(r"a", "X", "banana"))       # (result, kitni dafa)
res, n = re.subn(r"a", "X", "banana")
assert res == "bXnXnX" and n == 3
print()

# sub with FUNCTION repl - computed replacement
def double_num(mt):
    return str(int(mt.group()) * 2)
print("sub(func)   :", re.sub(r"\d+", double_num, "a1 b20 c300"))
assert re.sub(r"\d+", double_num, "a1 b20 c300") == "a2 b40 c600"

# Practical: masking sensitive data
def mask_card(mt):
    digits = re.sub(r"\D", "", mt.group())          # sirf digits rakho
    return "*" * (len(digits) - 4) + digits[-4:]    # aakhri 4 dikhao
print("masked cards:", re.sub(r"\b(?:\d[ -]?){13,16}\d\b", mask_card, "Card 4111-1111-1111-1111 ok"))

# Practical: template rendelaying
ctx = {"name": "Imran", "city": "Karachi"}
tpl = "Salam {{name}}, aap {{city}} from are. Missing: {{age}}"
print("rendelayed    :", re.sub(r"\{\{(\w+)\}\}", lambda m: str(ctx.get(m.group(1), "N/A")), tpl))
print()

# split
print("split \\s+       :", re.split(r"\s+", "a b   c\td"))
print("split multi-sep  :", re.split(r"[,;|]\s*", "a, b; c|d"))
print("split maxsplit=1 :", re.split(r",", "a,b,c", maxsplit=1))
print("split w/ GROUP   :", re.split(r"(\d)", "a1b2c"), " <-- separators bhi list me!")
print("split empty match:", re.split(r"", "abc"), " <-- 3.7+ zero-width split allowed")
assert re.split(r"(\d)", "a1b2c") == ["a", "1", "b", "2", "c"]

In [ ]:
# ------------------------------------------------------------------
# Compiled patterns: speed + reuse
# ------------------------------------------------------------------
lines = [f"user{i}@mail{i % 7}.com scored {i % 100}" for i in range(4000)]
PAT = r"(\w+)@([\w.]+) scored (\d+)"

t0 = perf_counter()
for ln in lines:
    re.search(PAT, ln)                 # module-level (internal cache hit is is)
t_module = perf_counter() - t0

compiled = re.compile(PAT)             # ek dafa parse
t0 = perf_counter()
for ln in lines:
    compiled.search(ln)                # method call - koi cache lookup not
t_compiled = perf_counter() - t0

print(f"lines            : {len(lines):,}")
print(f"re.search()      : {t_module*1000:7.2f} ms")
print(f"compiled.search(): {t_compiled*1000:7.2f} ms")
print(f"speedup          : ~{t_module/max(t_compiled, 1e-9):.2f}x (chhota, because re of internal cache is)")
print()
print("Compile to do of asli wajhein:")
print("  1. re.VERBOSE of sath readable, documented pattern")
print("  2. Module-level constant -> intent clean, hot loop me cache lookup not")
print("  3. Pattern object of methods: .search .match .findall .sub .split .finditer")
print("  4. pos/endpos arguments sirf compiled patterns but get are")
print()

# compiled pattern of extra tools
cp = re.compile(r"\d+")
print("pattern string  :", cp.pattern)
print("search(pos=5)   :", cp.search("12 34 56 78", 5))      # index 5 from ahead dhundo
print("groupindex      :", re.compile(r"(?P<a>x)(?P<b>y)").groupindex)
print()

# ------------------------------------------------------------------
# CATASTROPHIC BACKTRACKING - sirf padhne of for, CHALANA not
# ------------------------------------------------------------------
BAD_PATTERNS = [
    r"^(a+)+$",              # nested quantifier
    r"^(\w+\s?)*$",          # ambiguous: \w+ and \s? overlap
    r"^(.*,)*$",             # nested star
    r"^(a|a)*$",             # ambiguous alternation
]
print("KHATARNAAK patterns (this cell inhe chalata not):")
for p in BAD_PATTERNS:
    print("   ", p)
print()
print('Input "aaaa...ab" (match FAIL) but engine 2**n raaste try does is.')
print("40 a's = ~1e12 steps = minutes/hours. Yehi ReDoS attack is.")
print()

# SAFE versions - inhe chalana exactly theek is
SAFE_TEXT = "a" * 5000 + "b"           # 5001 chars, match fail hoga
safe_checks = [
    (r"^a+$",        "flat quantifier - koi nesting not"),
    (r"^[^b]*$",     "negated class - deterministic"),
]
for pat, why in safe_checks:
    t0 = perf_counter()
    ok = re.fullmatch(pat, SAFE_TEXT) is not None
    dt = (perf_counter() - t0) * 1000
    print(f"  {pat:<12} match={ok!s:<5} {dt:7.3f} ms   ({why})")

# Possessive/atomic bhi safe bana gives is (Python 3.11+)
try:
    t0 = perf_counter()
    ok = re.fullmatch(r"^(?>a+)+$", SAFE_TEXT) is not None
    print(f"  {'^(?>a+)+$':<12} match={ok!s:<5} {(perf_counter()-t0)*1000:7.3f} ms   (atomic group)")
except re.error:
    print("  atomic groups is Python me available not")
print()
print("Golden rules: nested quantifier no write | negated class prefer do |")
print("              user of regex trust no do | input length cap do.")

## D8. Real validations - email, Pakistani mobile, password, log parsing, dates

### Email
"Perfect" email regex **exist not does** (RFC 5322 of regex ~6000 chars of is).
Practical approach:
1. Ek **sane** regex from obvious ghalat cheezein rok do
2. Asli verification **confirmation email** bhej kar do
3. Production me `email.utils.parseaddr` / `email-validator` library use do

```python
r"^[\w.+-]+@[\w-]+(?:\.[\w-]+)+$"
```

### Pakistani mobile number
Formats jo log likhte are:
| Input | Valid? |
|---|---|
| `03001234567` | yes (local, 11 digits) |
| `0300-1234567` | yes (dash) |
| `+923001234567` | yes (E.164) |
| `+92 300 1234567` | yes (spaces) |
| `00923001234567` | yes (old international prefix) |
| `0300123456` | **not** (10 digits) |
| `04001234567` | **not** (mobile network code `3` from shuru is is) |

```python
r"^(?:(?:\+92|0092)[\s-]?|0)3\d{2}[\s-]?\d{7}$"
```

### Password strength
Lookaheads from AND-conditions (D5 me dekha). Plus **length limit** rakho (DoS from bachao)
and asli app me `zxcvbn` jaisi entropy library think - regex sirf minimum floor is.

### Log parsing
`re.VERBOSE` + **named groups** + `finditer` = maintainable log parser jo seedha `dict` gives is.

### HTML/XML - **regex from parse no do**
Regex nested/recursive structures handle not kar can (regular languages me nesting not is).
Attributes me `>` , comments, CDATA, unclosed tags - sab tod dete are.
**Use `html.parser`, `lxml`, ya `BeautifulSoup`.** Regex sirf **chhote, controlled**
extraction (like apni hi generated string from ek tag nikalna) of for theek is.

In [ ]:
# ------------------------------------------------------------------
# VALIDATION 1: Email  |  VALIDATION 2: Pakistani mobile number
# ------------------------------------------------------------------
EMAIL_RE = re.compile(r'''
    ^
    [A-Za-z0-9_.+-]+          # local part: letters, digits, . _ + -
    @
    [A-Za-z0-9-]+             # domain of pehla label
    (?:\.[A-Za-z0-9-]+)*      # optional sub-domains
    \.[A-Za-z]{2,}            # TLD: kam az kam 2 letters
    $
''', re.VERBOSE)

emails = [
    "imran.bizee@cyggs.tech",     # valid
    "ali+news@gmail.com",         # valid (plus addressing)
    "a_b@sub.domain.co.uk",       # valid (multi-level)
    "no-at-sign.com",             # invalid
    "double@@mail.com",           # invalid
    "trailing@dot.",              # invalid
    "spaces in@mail.com",         # invalid
    "short@d.c",                  # valid (2-letter TLD)
    "@nolocal.com",               # invalid
]
print("EMAIL VALIDATION")
for e in emails:
    print(f"  {e:<26} -> {'VALID' if EMAIL_RE.fullmatch(e) else 'invalid'}")

assert EMAIL_RE.fullmatch("imran.bizee@cyggs.tech")
assert EMAIL_RE.fullmatch("ali+news@gmail.com")
assert not EMAIL_RE.fullmatch("no-at-sign.com")
assert not EMAIL_RE.fullmatch("double@@mail.com")
assert not EMAIL_RE.fullmatch("trailing@dot.")
print()

# ------------------------------------------------------------------
PK_MOBILE_RE = re.compile(r'''
    ^
    (?:                       # country prefix of hissa
        (?:\+92|0092)[\s-]?   #   +92 ya 0092, ahead optional space/dash
      | 0                     #   ya local trunk '0'
    )
    3                         # mobile numbers hamesha 3 from shuru
    \d{2}                     # network code of baqi 2 digits (300, 321, 345...)
    [\s-]?                    # optional separator
    \d{7}                     # subscriber number: 7 digits
    $
''', re.VERBOSE)

phones = [
    "03001234567", "0321-1234567", "+923451234567", "+92 300 1234567",
    "00923331234567", "0300123456", "04001234567", "3001234567",
    "+91 300 1234567", "0300 1234 567",
]
print("PAKISTANI MOBILE VALIDATION")
for p in phones:
    print(f"  {p:<18} -> {'VALID' if PK_MOBILE_RE.fullmatch(p) else 'invalid'}")

assert PK_MOBILE_RE.fullmatch("03001234567")
assert PK_MOBILE_RE.fullmatch("0321-1234567")
assert PK_MOBILE_RE.fullmatch("+923451234567")
assert PK_MOBILE_RE.fullmatch("+92 300 1234567")
assert PK_MOBILE_RE.fullmatch("00923331234567")
assert not PK_MOBILE_RE.fullmatch("0300123456")      # 1 digit kam
assert not PK_MOBILE_RE.fullmatch("04001234567")     # 4 from shuru
assert not PK_MOBILE_RE.fullmatch("3001234567")      # koi prefix not
print()

# Normalize to do of helper (validation of after canonical form create)
def normalize_pk(phone: str) -> str | None:
    if not PK_MOBILE_RE.fullmatch(phone):
        return None
    digits = re.sub(r"\D", "", phone)                # sirf digits
    digits = re.sub(r"^(?:0092|92|0)", "", digits)   # prefix hatao -> 10 digits
    return "+92" + digits

for p in ["03001234567", "0321-1234567", "+92 300 1234567", "00923331234567", "bad"]:
    print(f"  normalize({p!r:<18}) -> {normalize_pk(p)}")
assert normalize_pk("03001234567") == "+923001234567"
assert normalize_pk("+92 300 1234567") == "+923001234567"
assert normalize_pk("bad") is None

In [ ]:
# ------------------------------------------------------------------
# VALIDATION 3: Password strength (score + reasons)
# ------------------------------------------------------------------
PW_RULES = [
    (r".{8,}",        "kam az kam 8 characters"),
    (r"[a-z]",        "ek lowercase letter"),
    (r"[A-Z]",        "ek uppercase letter"),
    (r"\d",           "ek digit"),
    (r"[^A-Za-z0-9]", "ek special character"),
]
NO_SPACE   = re.compile(r"^\S+$")
NOT_COMMON = {"password", "12345678", "qwerty123", "pakistan"}

def check_password(pw: str) -> dict:
    '''Rules of score + missing reasons of list gives is.'''
    missing = [why for pat, why in PW_RULES if not re.search(pat, pw)]
    if not NO_SPACE.fullmatch(pw):
        missing.append("whitespace not hona should")
    if pw.lower() in NOT_COMMON:
        missing.append("bohat aam password is")
    if len(pw) > 128:
        missing.append("128 chars from lamba not (DoS guard)")
    score = len(PW_RULES) - len([m for m in missing if m in [w for _, w in PW_RULES]])
    label = ["very weak", "weak", "fair", "good", "strong", "very strong"][score]
    return {"ok": not missing, "score": score, "label": label, "missing": missing}

for pw in ["Pak@1234", "abc", "alllowercase1!", "NoDigits!!", "Has Space1@", "Password", "Karachi#2024Rocks"]:
    r = check_password(pw)
    flag = "OK  " if r["ok"] else "FAIL"
    print(f"  {flag} {pw!r:<20} score={r['score']}/5 ({r['label']})")
    if r["missing"]:
        print(f"        missing -> {', '.join(r['missing'])}")

assert check_password("Pak@1234")["ok"] is True
assert check_password("abc")["ok"] is False
assert "whitespace not hona should" in check_password("Has Space1@")["missing"]
assert check_password("Karachi#2024Rocks")["score"] == 5
print()

# Single-regex version (lookaheads) - chhota, magar "what missing is" not bata can
ONE_SHOT = re.compile(r"^(?=.*[a-z])(?=.*[A-Z])(?=.*\d)(?=.*[^A-Za-z0-9])(?!.*\s).{8,128}$")
print("ONE_SHOT 'Pak@1234' :", bool(ONE_SHOT.fullmatch("Pak@1234")))
print("ONE_SHOT 'weakpass' :", bool(ONE_SHOT.fullmatch("weakpass")))
print()
print("Interview tip: rule-list that approach production me behtar is because user then")
print("actionable feedback gives is; single regex sirf yes/no gives is.")

In [ ]:
# ------------------------------------------------------------------
# LOG PARSING -> list[dict]   +   DATE EXTRACTION   +   HTML caution
# ------------------------------------------------------------------
ACCESS_LOG = '''\
203.0.113.7 - imran [15/Mar/2024:13:55:36 +0500] "GET /api/ordelays?page=2 HTTP/1.1" 200 2326 "-" "curl/8.4.0"
198.51.100.22 - - [15/Mar/2024:13:55:40 +0500] "POST /api/login HTTP/1.1" 401 128 "-" "Mozilla/5.0"
203.0.113.7 - imran [15/Mar/2024:13:56:02 +0500] "GET /static/app.css HTTP/1.1" 304 - "-" "Mozilla/5.0"
BROKEN LINE THAT SHOULD NOT PARSE
192.0.2.9 - admin [15/Mar/2024:14:01:11 +0500] "DELETE /api/ordelays/88 HTTP/1.1" 500 512 "-" "PostmanRuntime/7.36"
'''

# Combined Log Format of parser: VERBOSE + named groups = maintainable
LOG_RE = re.compile(r'''
    ^(?P<ip>\d{1,3}(?:\.\d{1,3}){3})      \s+   # IPv4
    (?P<ident>\S+)                        \s+   # identd (usually '-')
    (?P<user>\S+)                         \s+   # authenticated user ya '-'
    \[(?P<ts>[^\]]+)\]                    \s+   # [15/Mar/2024:13:55:36 +0500]
    "(?P<method>[A-Z]+)\s+
     (?P<path>\S+)\s+
     (?P<proto>HTTP/[\d.]+)"              \s+   # request line
    (?P<status>\d{3})                     \s+   # status code
    (?P<size>\d+|-)                       \s+   # bytes ya '-'
    "(?P<referer>[^"]*)"                  \s+
    "(?P<agent>[^"]*)"$
''', re.VERBOSE)

parsed, bad = [], []
for line in ACCESS_LOG.strip().splitlines():
    mt = LOG_RE.match(line)
    if not mt:
        bad.append(line)
        continue
    d = mt.groupdict()
    d["status"] = int(d["status"])                             # types theek do
    d["size"] = 0 if d["size"] == "-" else int(d["size"])
    d["ts"] = datetime.strptime(d["ts"], "%d/%b/%Y:%H:%M:%S %z")   # tz-aware datetime
    parsed.append(d)

print(f"parsed {len(parsed)} lines, {len(bad)} rejected")
for d in parsed:
    print(f"  {d['ts']:%H:%M:%S} {d['ip']:<14} {d['method']:<6} {d['status']} {d['size']:>5}B  {d['path']}")
print("rejected:", bad)
assert len(parsed) == 4 and len(bad) == 1
assert parsed[0]["status"] == 200 and parsed[2]["size"] == 0
assert parsed[0]["ts"].tzinfo is not None
print()

# ------------------------------------------------------------------
# DATE EXTRACTION - kai formats ek pattern me
# ------------------------------------------------------------------
doc = ("Invoice dated 2024-03-15 was due 15/04/2024, remindelay sent on Apr 20, 2024 "
       "and finally paid 2024-05-01. Bogus date 2024-13-45 bhi likha is.")

DATE_RE = re.compile(r'''
      (?P<iso>\d{4}-\d{2}-\d{2})                          # 2024-03-15
    | (?P<dmy>\d{1,2}/\d{1,2}/\d{4})                      # 15/04/2024
    | (?P<mdy>(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)
       [a-z]*\s+\d{1,2},\s*\d{4})                         # Apr 20, 2024
''', re.VERBOSE)

print("DATE EXTRACTION")
FORMATS = {"iso": "%Y-%m-%d", "dmy": "%d/%m/%Y", "mdy": "%b %d, %Y"}
found = []
for mt in DATE_RE.finditer(doc):
    kind = mt.lastgroup                       # kaunsa named group match happened
    raw = mt.group()
    try:
        # normalize -> real date object. Bogus dates yahan pakre go are (regex from not)
        clean = re.sub(r"\s+", " ", raw.replace(",", ", ").replace(",  ", ", "))
        parsed_date = datetime.strptime(raw if kind != "mdy" else clean[:12].strip(), FORMATS[kind]).date()
        found.append(parsed_date)
        print(f"  {raw:<16} ({kind}) -> {parsed_date}")
    except Valueerror as e:
        print(f"  {raw:<16} ({kind}) -> INVALID: {e}")

print("Sabaq: regex SHAPE check does is, VALIDITY not. 2024-13-45 shape me sahi is")
print("       magar strptime use reject does is. Dono milakar use do.")
assert date(2024, 3, 15) in found and date(2024, 4, 15) in found
print()

# ------------------------------------------------------------------
# HTML: regex of hadd
# ------------------------------------------------------------------
tricky = '<a href="x>y.html" title="not > a tag">Click</a><!-- <b>comment</b> -->'
print("HTML CAUTION")
print("  naive <.*?>  :", re.findall(r"<.*?>", tricky))
print("  ^ attribute of inside '>' ne pattern tod diya. Nested/comment/CDATA bhi todte are.")
print()
print("  SAHI tareeqa: stdlib html.parser (ya lxml / BeautifulSoup)")
from html.parser import HTMLParser
class TagCollector(HTMLParser):
    def __init__(self):
        super().__init__(); self.tags = []; self.text = []
    def handle_starttag(self, tag, attrs): self.tags.append((tag, dict(attrs)))
    def handle_data(self, data):
        if data.strip(): self.text.append(data.strip())

tc = TagCollector(); tc.feed(tricky)
print("  parser tags  :", tc.tags)
print("  parser text  :", tc.text)
assert tc.tags[0][0] == "a" and tc.tags[0][1]["href"] == "x>y.html"
print()
print("  Regex sirf tab theek is jab input CHHOTA and CONTROLLED is, e.g. tag strip:")
print("  strip tags   :", re.sub(r"<[^>]+>", "", "<p>Hello <b>World</b></p>"))